# mini-engine benchmarks (M5)
Runtime → Change runtime type → a GPU (L4 or A100 preferred, T4 works). Then Runtime → Run all.
Each system runs in its own process. Results: `benchmarks/results/*.json` and the summary table below.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!rm -rf /content/me && git clone -q -b feat/m5-speculative https://github.com/YuvGev00/mini-engine /content/me
%cd /content/me
!pip install -q vllm
!pip install -q -e . --no-deps
!python -c "import vllm, torch, transformers; print('vllm', vllm.__version__, 'torch', torch.__version__, 'transformers', transformers.__version__)"

In [ ]:
# Correctness on the GPU before speed: our greedy output vs HF on the real model (agreement is also in the summary).
!cd /content/me && python benchmarks/bench.py --system mini --out benchmarks/results/mini.json 2>&1 | grep -v -i warn | tail -12

In [ ]:
!cd /content/me && python benchmarks/bench.py --system hf --out benchmarks/results/hf.json 2>&1 | grep -v -i warn | tail -8

In [ ]:
!cd /content/me && python benchmarks/bench.py --system vllm --out benchmarks/results/vllm.json 2>&1 | grep -E '^vllm |Error|error' | tail -12

In [ ]:
!cd /content/me && python benchmarks/summarize.py benchmarks/results

In [ ]:
# Compact copy of the raw results (metrics only), for committing to the repo.
import json, glob
for p in sorted(glob.glob('/content/me/benchmarks/results/*.json')):
    d = json.load(open(p))
    for r in d['rows']:
        r.pop('outputs_head', None)
    print('RESULT', p.split('/')[-1], json.dumps(d, separators=(',', ':')))